# Tiền xử lý dữ liệu (Phiên bản cải tiến)

Notebook này kế thừa quy trình tiền xử lý từ file `02-data-preprocessing.ipynb` gốc và **cải tiến** theo hai góp ý quan trọng:

1. **Số liệu:** Thay vì xóa hoàn toàn các con số, quy đổi chúng thành token chuẩn `__number__` để mô hình học được đặc trưng "xuất hiện số" (email spam thường chứa số tiền lớn, tỷ lệ giảm giá,...).
2. **Dấu câu đặc biệt:** Giữ lại các ký tự có tín hiệu phân loại cao như `!` và `$` dưới dạng token `__exclamation__` và `__dollar__`, vì email spam thường lạm dụng các ký tự này.

## 1. Cài đặt và khai báo thư viện

In [ ]:
%pip install nltk -q

In [ ]:
import pandas as pd
import re, nltk, string, ssl
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

### Fix lỗi SSL trên macOS

macOS không tự động tin tưởng chứng chỉ SSL hệ thống khi Python gọi HTTPS. Đoạn code dưới đây tạm bỏ qua xác thực SSL để `nltk.download()` hoạt động bình thường.

In [ ]:
# Fix SSL Certificate Verify Failed trên macOS
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

## 2. Đọc dữ liệu thô từ file gốc

In [ ]:
raw_dataset = pd.read_csv('../data/raw/enron_spam_data.csv')
print(f"Tổng số bản ghi ban đầu: {len(raw_dataset)}")
raw_dataset.info()

## 3. Xử lý giá trị khuyết thiếu và loại bỏ trùng lặp

In [ ]:
# Điền chuỗi rỗng cho các giá trị NaN ở cột Subject và Message
raw_dataset['Subject'] = raw_dataset['Subject'].fillna('')
raw_dataset['Message'] = raw_dataset['Message'].fillna('')

print(f"Số bản ghi trước khi loại bỏ trùng lặp: {len(raw_dataset)}")

# Loại bỏ các bản ghi trùng lặp hoàn toàn cả Subject và Message (phát hiện từ EDA)
raw_dataset = raw_dataset.drop_duplicates(subset=['Subject', 'Message'], keep='first')

print(f"Số bản ghi sau khi loại bỏ trùng lặp: {len(raw_dataset)}")

## 4. Định nghĩa các hàm tiện ích cho tiền xử lý

Hàm `get_wordnet_pos` chuyển đổi nhãn POS Tag của NLTK sang định dạng WordNet để phục vụ Lemmatization chính xác theo ngữ cảnh.

In [ ]:
STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    """Chuyển đổi nhãn POS Tag sang định dạng WordNet."""
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

## 5. Hàm tiền xử lý văn bản cải tiến

### Các cải tiến so với phiên bản gốc:

- **Cải tiến 1 - Xử lý số:** Thay vì xóa hoàn toàn các chuỗi số (`r'\b\d+\b'` → rỗng), giờ quy đổi chúng thành token `__number__`. Điều này giúp mô hình Naive Bayes học được rằng email spam thường chứa nhiều con số (số tiền thưởng, tỷ lệ giảm giá,...).

- **Cải tiến 2 - Giữ dấu câu đặc biệt:** Trước khi xóa dấu câu, trích xuất và đếm tần suất các ký tự `!` và `$`. Nếu xuất hiện, chèn token `__exclamation__` hoặc `__dollar__` vào chuỗi kết quả. Email spam thường lạm dụng các ký tự này (`!!!`, `$$$`) để gây chú ý.

In [ ]:
def preprocessing_message(text: str) -> str:
    """Tiền xử lý một chuỗi văn bản email (phiên bản cải tiến)."""
    # Xử lý giá trị NaN
    cleaned = '' if pd.isna(text) else str(text)

    # Loại bỏ ký tự Unicode phi chuẩn
    cleaned = cleaned.encode('ascii', 'ignore').decode('ascii')

    # Loại bỏ thẻ HTML
    cleaned = re.sub(r'<[^>]+>', ' ', cleaned)

    # Thay thế các URL thành token __url__
    cleaned = re.sub(
        r'(https?://\S+|ftp://\S+|www\.\S+)',
        '__url__', cleaned
    )

    # === CẢI TIẾN 1: Quy đổi số thành token __number__ thay vì xóa hoàn toàn ===
    cleaned = re.sub(r'\b\d+\b', ' __number__ ', cleaned)

    # === CẢI TIẾN 2: Trích xuất dấu câu đặc biệt trước khi xóa dấu câu ===
    # Đếm tần suất dấu ! và $ để tạo token tương ứng
    exclamation_count = cleaned.count('!')
    dollar_count = cleaned.count('$')

    # Xóa dấu câu thông thường (nhưng giữ lại __ cho các token đặc biệt)
    # Tạo tập dấu câu cần xóa: loại bỏ _ khỏi danh sách để giữ nguyên các token __xxx__
    punct_to_remove = string.punctuation.replace('_', '')
    cleaned = re.sub(f'[{re.escape(punct_to_remove)}]', ' ', cleaned)

    # Chuẩn hóa khoảng trắng (xóa \n, \t, khoảng trắng thừa)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Tách từ (Tokenization)
    tokens = nltk.word_tokenize(cleaned)

    # Gán nhãn từ loại (POS Tagging)
    pos_tags = nltk.pos_tag(tokens)

    # Lemmatization + Lọc từ dừng + Lọc từ quá ngắn
    final_tokens = []
    for word, tag in pos_tags:
        # Giữ nguyên các token đặc biệt (__url__, __number__)
        if word.startswith('__') and word.endswith('__'):
            final_tokens.append(word)
            continue

        wn_tag = get_wordnet_pos(tag)
        lemma = lemmatizer.lemmatize(word, pos=wn_tag)
        lemma_lower = lemma.lower().strip()

        if lemma_lower not in STOP_WORDS and len(lemma_lower) > 2:
            final_tokens.append(lemma_lower)

    # === CẢI TIẾN 2 (tiếp): Chèn token dấu câu đặc biệt vào cuối ===
    if exclamation_count > 0:
        final_tokens.append('__exclamation__')
    if dollar_count > 0:
        final_tokens.append('__dollar__')

    return ' '.join(final_tokens)

## 6. Áp dụng tiền xử lý lên toàn bộ dữ liệu

In [ ]:
preprocessed_dataset = pd.DataFrame()

preprocessed_dataset['Subject'] = raw_dataset['Subject'].apply(preprocessing_message)
preprocessed_dataset['Message'] = raw_dataset['Message'].apply(preprocessing_message)
preprocessed_dataset['Label'] = raw_dataset['Spam/Ham'].values

print(f"Số bản ghi sau khi áp dụng tiền xử lý: {len(preprocessed_dataset)}")

## 7. Loại bỏ các bản ghi trống hoàn toàn sau tiền xử lý

In [ ]:
before_count = len(preprocessed_dataset)

# Loại bỏ các dòng mà cả Subject và Message đều rỗng sau khi xử lý
preprocessed_dataset = preprocessed_dataset[
    (preprocessed_dataset['Subject'] != '') | (preprocessed_dataset['Message'] != '')
]

after_count = len(preprocessed_dataset)
print(f"Đã loại bỏ {before_count - after_count} bản ghi trống. Còn lại: {after_count} bản ghi.")

## 8. Xem trước dữ liệu sau tiền xử lý

In [ ]:
preprocessed_dataset.info()
print('\n')
print(preprocessed_dataset['Label'].value_counts())
print('\n--- Mẫu dữ liệu đã xử lý ---')
pd.set_option('display.max_colwidth', 120)
preprocessed_dataset.head(10)

## 9. Kiểm tra nhanh các token đặc biệt đã được chèn đúng

In [ ]:
# Kết hợp Subject và Message để đếm token
all_text = preprocessed_dataset['Subject'] + ' ' + preprocessed_dataset['Message']

url_count = all_text.str.count('__url__').sum()
number_count = all_text.str.count('__number__').sum()
excl_count = all_text.str.count('__exclamation__').sum()
dollar_count = all_text.str.count('__dollar__').sum()

print(f"Tổng số token __url__         : {int(url_count):,}")
print(f"Tổng số token __number__      : {int(number_count):,}")
print(f"Tổng số token __exclamation__ : {int(excl_count):,}")
print(f"Tổng số token __dollar__      : {int(dollar_count):,}")

## 10. Xuất dữ liệu đã xử lý ra file CSV

In [ ]:
import os

output_dir = './data/preprocessed'
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, 'enron_spam_data_preprocessed.csv')
preprocessed_dataset.to_csv(output_path, index=False)

print(f"Đã xuất dữ liệu sạch ra: {output_path}")
print(f"Kích thước file: {os.path.getsize(output_path) / (1024*1024):.2f} MB")